In [ ]:
import pandas as pd
import re


In [ ]:
# Open our data + basic info

# utf-8 causes decoding error
df = pd.read_csv('../data/raw/shark_attacks_raw.csv',encoding='latin1')
print(df.shape)
print(df.info())    

In [ ]:
# Percent of empty per column
missing_percent = df.isnull().mean().sort_values(ascending=False) * 100
print("Percentage of missing values per column:")
print(missing_percent)

In [ ]:
# Delete empty columns
df = df.drop(columns=['Unnamed: 22', 'Unnamed: 23'])

#Accuracy down to the day of the month is unlikely to be necessary; we'll single out the month, as statistics on attacks depending on the season may be of interest
df['Month'] = pd.to_datetime(df['Date'], errors='coerce').dt.month

# Delete date
df = df.drop(columns=['Date'])

print(f"After deleting empty columns: {df.shape}")

In [ ]:
necessary_columns = ['Fatal (Y/N)','Month', 'Case Number','Country','Area', 'Type', 'Activity']

# Delete rows where all the main characteristics are empty

df = df.dropna(subset=necessary_columns, how='all')
print(f"After deleting rows with empty main characteristics: {df.shape}")

# Percent of empty per column after cleaning
print(df.isnull().mean().sort_values(ascending=False) * 100)

In [ ]:
# Fatal go from Y/N represent to binar 1/0 
def fatal_to_binary(value):
    if pd.isna(value):
        return None
    if value == "Y":
        return 1
    if value == "N":
        return 0
    else:
        return None

df['Fatal'] = df['Fatal (Y/N)'].apply(fatal_to_binary)


print(df[['Fatal (Y/N)', 'Fatal']].head(10))
print(df['Fatal'].value_counts(dropna=False))

# Get rid of Fatal(Y/N) because now we have clear binary view
df = df.drop(columns=['Fatal (Y/N)'])


In [ ]:
# Rename column 'Species ' to 'Species'
df = df.rename(columns={'Species ': 'Species'})



# Sex to binary where f = 1, m = 0

def sex_to_binary(value):
    if pd.isna(value):
        return None
    val = str(value).lower()
    if "f" in val:
        return 1
    if "m" in val:
        return 0
    else:
        return None

df['Sex'] = df['Sex '].apply(sex_to_binary)


print(df[['Sex', 'Sex ']].head(10))
print(df['Sex'].value_counts(dropna=False))

# Get rid of Fatal(Y/N) because now we have clear binary view
df = df.drop(columns=['Sex '])


In [ ]:
# text columns to make lower case plus clear excess spaces
text_columns = ['Case Number', 'Type', 'Country', 'Area', 'Location', 'Activity', 'Name', 'Injury', 'Species', "Time"]

for column in text_columns:
        # Convert to string, lowercase, strip whitespace
        df[column] = df[column].astype(str).str.lower().str.strip()
        # Replace common empty string representations with NaN
        df[column] = df[column].replace(['nan', 'none', '', 'unknown'], None)


In [ ]:
# Turning time to Time category


# getting hour from different time formats with numbers in them

def to_military_time(hour,ampm):
    if 'a' in ampm:
        if hour != 12:
            return hour
        else:
            return 0
    else:
        if hour != 12:
            return hour + 12
        else:
            return 12

def extract_hour(value):
    if pd.isna(value):
        return None
    value = str(value).lower().strip()
    # Pattern for getting an hour from formats that include numbers in them
    patterns = [
        (r'(\d{1,2})(?:h|:)(?:\d{2})?\s*(am|pm|a\.m\.|p\.m\.)', True),
        (r'(\d{1,2})\s*(am|pm|a\.m\.|p\.m\.)', True),
        (r'(\d{1,2})(?:h|:|\d{2})', False),
        (r'(\d{1})(\d{2})', False)      
    ]
    for pattern, has_ampm in patterns:
        match = re.search(pattern, value)
        if match:
            hour = int(match.group(1))
            
            if has_ampm:
                ampm = match.group(2).lower()
                hour = to_military_time(hour, ampm)

            # Is time corret?
            if 0 <= hour <= 23:
                return hour 
    
    return None

# Working with words meaning time in Time column
def categorize_by_words(value):
    if pd.isna(value):
        return None
    value = str(value).lower().strip()
    keywords = {
        'late night': ['after midnight', 'before dawn', 'wee hours', 'early morning hours'],
        'dawn': ['dawn', 'daybreak', 'sunrise', 'first light', 'before sunrise'],
        'morning': ['morning', 'am', 'a.m.', 'early morning', 'late morning', 'mid morning', 'mid-morning'],
        'afternoon': ['afternoon', 'pm', 'p.m.', 'after noon', 'mid afternoon', 'early afternoon', 'late afternoon'],
        'midday': ['midday', 'noon', 'lunchtime', 'just before noon', 'just after 12h00', '12h00'],
        'evening': ['evening', 'dusk', 'sunset', 'early evening', 'just before dusk', 'after dusk', 'nightfall'],
        'night': ['night', 'dark', 'after dark', 'late night', 'midnight']
    }

    for category, words in keywords.items():
        for word in words:
            if word in value:
                return category
    
    return None

# All in one function
def classify_time(value):
    if pd.isna(value):
        return None
    
    value = str(value).strip()
    
    keyword_category = categorize_by_words(value)
    if keyword_category:
        return keyword_category
    
    hour = extract_hour(value)

    if hour:
        if 4 <= hour < 7:
            return 'dawn'
        elif 7 <= hour < 12:
            return 'morning'
        elif 12 <= hour < 14:
            return 'midday'
        elif 14 <= hour < 17:
            return 'afternoon'
        elif 17 <= hour < 21:
            return 'evening'
        elif 21 <= hour < 24:
            return 'night'
        elif 0 <= hour < 4:
            return 'late night'
    
    return None

df['Time Category'] = df['Time'].apply(classify_time)

# Delete Time column
df = df.drop(columns=['Time'])

In [ ]:
# If year looks incorrect set it as NaN
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
df['Year'] = df['Year'].apply(lambda x: x if 1500 <= x <= 2026 else None)

In [ ]:
# Species column super messy, has sizes in it and some other thing. maling it more categorized - some common species plus sharks plus none if not a shark 
def categorize_shark(value):
    if pd.isna(value):
        return None
    value = str(value).lower().strip()
    if 'whitetip' in value or 'white tip' in value:
        return 'whitetip'
    if 'white' in value:
        return 'white'
    if 'tiger' in value:
        return 'tiger'
    if 'bull' in value:
        return 'bull'
    if 'hammerhead' in value:
        return 'hammerhead'
    if 'mako' in value:
        return 'mako'
    if 'blue' in value:
        return 'blue'
    if 'lemon' in value:
        return 'lemon'
    if 'shark' in value:
        return 'other'
    return None

df['Species'] = df['Species'].apply(categorize_shark)

# Сколько в каждой категории
print(df['Species'].value_counts())

In [ ]:
# Save cleaned dataset
df.to_csv('C:/my pet projects/shark_attacks/data/processed/shark_attacks_clean.csv', index=False)
print("\n=== Cleaned dataset saved to processed/ ===")